In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:22px;}
</style>
"""))

In [1]:
import pandas as pd # 파일입력(read_excel), 교차표(crosstab), 원핫인코딩(get_dummies)
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split # 훈련셋과 시험셋 분리 함수
from tensorflow.keras.utils import to_categorical # 원핫인코딩
from tensorflow.keras.models import Sequential, save_model, load_model
# 과적합을 줄이기 위한 방법 : dropout, 배치정규화, L2정규화
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
# 데이터 불균형 극복 5.5 : 4.5 => 5:5
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt # 모델 학습과정 살펴보기
from sklearn.metrics import confusion_matrix # 혼동행렬

```
age: 나이 (년 단위)
sex: 성별 (1 = 남성, 0 = 여성)
cp (Chest Pain Type): 흉통의 유형 (0~3 값, 일반 협심증부터 무증상까지 분류)
trestbps: 안정 시 혈압 (mmHg)
chol: 혈청 콜레스테롤 수치 (mg/dl)
fbs (Fasting Blood Sugar): 공복 혈당이 120 mg/dl 초과인지 여부 (1 = 참, 0 = 거짓)
restecg: 안정 시 심전도(ECG) 결과 (0~2)
thalach: 운동 시 달성한 최대 심박수
exang: 운동 유발 협심증 여부 (1 = 있음, 0 = 없음)
oldpeak: 휴식기에 비해 운동 시 나타나는 ST 부위 하강(ST depression) 정도
slope: 최고 운동 시 ST 세그먼트의 기울기ca: 형광투시법으로 색칠된 주요 혈관 수 (0~3개)
thal: 지중해빈혈 등 혈액 질환 여부 (정상, 고정 결함, 가역적 결함 등)
target (Diagnosis): 심장질환 진단 결과 (0 = 정상, 1 = 심장질환 있음)
```

# 이진분류(로지스틱 회귀분석)
- 1. 데이터셋 생성 & 전처리
    * 엑셀 -> 데이터프레임 -> ?처리(결측치로 전환하여 결측치처리) -> X, y분리
        -> X변수의 scale조정 -> train_test_split()을 이용하여 학습셋과 테스트셋을 분리
- 2. 모델 생성(입력13,타겟1) - 과적합 고려 & 학습과정설정 & 학습
- 3. 모델 평가(그래프, 평가, 혼동행렬=교차표)
- 4. 모델 사용

## 1. 데이터셋 생성 & 전처리

In [2]:
df = pd.read_excel('data/heart-disease.xlsx',
                  # sheet_name='processed.cleveland'
                  )
df.head()

,age,sex,cp,treshtbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,hsl,heartDisease
0,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0
1,67,1,4,160,286,0,0,108,1,1.5,2,3,3,1
2,67,1,4,120,?,0,2,129,1,2.6,2,2,7,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   age           303 non-null    int64  
 1   sex           303 non-null    int64  
 2   cp            303 non-null    int64  
 3   treshtbps     303 non-null    int64  
 4   chol          303 non-null    object 
 5   fbs           303 non-null    int64  
 6   restecg       303 non-null    int64  
 7   thalach       303 non-null    int64  
 8   exang         303 non-null    int64  
 9   oldpeak       303 non-null    float64
 10  slope         303 non-null    int64  
 11  ca            303 non-null    object 
 12  hsl           303 non-null    object 
 13  heartDisease  303 non-null    int64  
dtypes: float64(1), int64(10), object(3)
memory usage: 33.3+ KB


In [12]:
df.isnull().sum()

age             0
sex             0
cp              0
treshtbps       0
chol            0
fbs             0
restecg         0
thalach         0
exang           0
oldpeak         0
slope           0
ca              0
hsl             0
heartDisease    0
dtype: int64

In [14]:
df.isin(['?']).sum()

age             0
sex             0
cp              0
treshtbps       0
chol            1
fbs             0
restecg         0
thalach         0
exang           0
oldpeak         0
slope           0
ca              4
hsl             2
heartDisease    0
dtype: int64

In [5]:
df[(df['chol']=='?') | (df['ca']=='?') | (df['hsl']=='?') ]

,age,sex,cp,treshtbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,hsl,heartDisease
2,67,1,4,120,?,0,2,129,1,2.6,2,2,7,1
87,53,0,3,128,216,0,2,115,0,0.0,1,0,?,0
166,52,1,3,138,223,0,0,169,0,0.0,1,?,3,0
192,43,1,4,132,247,1,2,143,1,0.1,2,?,7,1
266,52,1,4,128,204,1,0,156,1,1.0,2,0,?,1
287,58,1,2,125,220,0,0,144,0,0.4,2,?,7,0
302,38,1,3,138,175,0,0,173,0,0.0,1,?,3,0


In [3]:
# 방법1 : ?가 있는 행을 삭제
drop_idx = df[(df['chol']=='?') | (df['ca']=='?') | (df['hsl']=='?') ].index # 삭제할 행
df = df.drop(drop_idx)

In [9]:
# 방법2:?를 결측치로 대체 => 결측치 삭제
df = df.replace('?', np.nan)
df[(df['chol']=='?') | (df['ca']=='?') | (df['hsl']=='?') ]

,age,sex,cp,treshtbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,hsl,heartDisease


In [27]:
# 결측치가 포함된 데이터 추출
df[df['chol'].isna() | df['ca'].isna() | df['hsl'].isna()]
df[df.isna().any(axis=1)]

,age,sex,cp,treshtbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,hsl,heartDisease
2,67,1,4,120,NaN,0,2,129,1,2.6,2,2.0,7.0,1
87,53,0,3,128,216.0,0,2,115,0,0.0,1,0.0,NaN,0
166,52,1,3,138,223.0,0,0,169,0,0.0,1,NaN,3.0,0
192,43,1,4,132,247.0,1,2,143,1,0.1,2,NaN,7.0,1
266,52,1,4,128,204.0,1,0,156,1,1.0,2,0.0,NaN,1
287,58,1,2,125,220.0,0,0,144,0,0.4,2,NaN,7.0,0
302,38,1,3,138,175.0,0,0,173,0,0.0,1,NaN,3.0,0


In [31]:
# 결측치 처리 : 삭제(dropna) cf. fillna() apply()
df.dropna(how='any', inplace=True) # 결측치가 한열이라도 있으면 삭제
df[df.isna().any(axis=1)]

,age,sex,cp,treshtbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,hsl,heartDisease


In [10]:
df.isna().sum()

age             0
sex             0
cp              0
treshtbps       0
chol            0
fbs             0
restecg         0
thalach         0
exang           0
oldpeak         0
slope           0
ca              0
hsl             0
heartDisease    0
dtype: int64

In [11]:
df.shape

(296, 14)

In [12]:
# 타겟변수 분포가 균형적인지
df['heartDisease'].value_counts()/df.shape[0]

0    0.540541
1    0.459459
Name: heartDisease, dtype: float64

In [13]:
df['heartDisease'].value_counts(normalize=True)

0    0.540541
1    0.459459
Name: heartDisease, dtype: float64

In [4]:
# X, y분리
X = df.iloc[:, :-1].values # 맨마지막 열을 제외한 부분을 numpy배열
y = df.iloc[:, -1:].to_numpy() # 2차원 numpy 배열
X.shape, y.shape

((296, 13), (296, 1))

In [5]:
# X변수의 scale조정
scaler = MinMaxScaler()
scaled_X = scaler.fit_transform(X)
print('원 데이터 :', X[0])
print('스케일 조정 데이터 :', scaled_X[0])

원 데이터 : [63 1 1 145 233 1 2 150 0 2.3 3 0 6]
스케일 조정 데이터 : [0.70833333 1.         0.         0.48113208 0.24429224 1.
 1.         0.60305344 0.         0.37096774 1.         0.
 0.75      ]


In [6]:
# scaled_X와 y를 학습데이터셋:테스트셋 = 8:2
X_train, X_test, y_train, y_test = train_test_split(scaled_X,
                                    y,
                                    #train_size=0.8,
                                    test_size=0.2,
                                    random_state=7, # seed값
                                    stratify=y) # 층화추출

In [17]:
# 심장병 음성/양성 비율(y, y_train, y_test)
print(pd.DataFrame(y).value_counts(normalize=True))
print(pd.DataFrame(y_train).value_counts(normalize=True))
print(pd.DataFrame(y_test).value_counts(normalize=True))

0    0.540541
1    0.459459
dtype: float64
0    0.542373
1    0.457627
dtype: float64
0    0.533333
1    0.466667
dtype: float64


In [7]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((236, 13), (236, 1), (60, 13), (60, 1))

## 2. 모델생성 & 학습과정 설정 & 학습
- 모델 : 13 -> 32 -> 16 -> 8 -> 1
- 학습과정을 시각화한 그래프를 보고 과적합 줄이기
- 과적합 줄이기 위한 방법

In [9]:
model = Sequential(name='sequential')
model.add(Dense(units=32, input_dim=13, activation='relu'))
model.add(Dense(units=16, activation='relu'))
model.add(Dense(units=8, activation='relu'))
model.add(Dense(units=1, activation='sigmoid'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_4 (Dense)             (None, 32)                448       
                                                                 
 dense_5 (Dense)             (None, 16)                528       
                                                                 
 dense_6 (Dense)             (None, 8)                 136       
                                                                 
 dense_7 (Dense)             (None, 1)                 9         
                                                                 
Total params: 1,121
Trainable params: 1,121
Non-trainable params: 0
_________________________________________________________________


In [12]:
# 학습과정 설정
from tensorflow.keras.metrics import Recall, Precision, Accuracy
from tensorflow.keras.optimizers import Adam
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              # optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy', #Accuracy(name='accuracy'), 
                       Recall(name='recall'),        # 재현율(실제값 중심=1을 얼마나 잘 예측했는지)
                       Precision(name='precision')]) # 정밀도(예측값 중심=1이라고 예측한 것 중 얼마나 1인지)

In [13]:
# 학습
hist = model.fit(X_train, y_train, # 학습데이터
                epochs=200, # 학습횟수
                verbose=2)

Epoch 1/200
8/8 - 2s - loss: 0.6951 - accuracy: 0.5212 - recall: 0.6667 - precision: 0.4832 - 2s/epoch - 288ms/step
Epoch 2/200
8/8 - 0s - loss: 0.6810 - accuracy: 0.5763 - recall: 0.7407 - precision: 0.5263 - 47ms/epoch - 6ms/step
Epoch 3/200
8/8 - 0s - loss: 0.6677 - accuracy: 0.6229 - recall: 0.7315 - precision: 0.5683 - 58ms/epoch - 7ms/step
Epoch 4/200
8/8 - 0s - loss: 0.6538 - accuracy: 0.6864 - recall: 0.7593 - precision: 0.6308 - 42ms/epoch - 5ms/step
Epoch 5/200
8/8 - 0s - loss: 0.6405 - accuracy: 0.6822 - recall: 0.8056 - precision: 0.6170 - 50ms/epoch - 6ms/step
Epoch 6/200
8/8 - 0s - loss: 0.6269 - accuracy: 0.6695 - recall: 0.8148 - precision: 0.6027 - 41ms/epoch - 5ms/step
Epoch 7/200
8/8 - 0s - loss: 0.6133 - accuracy: 0.6780 - recall: 0.8241 - precision: 0.6096 - 26ms/epoch - 3ms/step
Epoch 8/200
8/8 - 0s - loss: 0.5993 - accuracy: 0.6992 - recall: 0.8426 - precision: 0.6276 - 49ms/epoch - 6ms/step
Epoch 9/200
8/8 - 0s - loss: 0.5850 - accuracy: 0.7246 - recall: 0.8241 

8/8 - 0s - loss: 0.2914 - accuracy: 0.8729 - recall: 0.8241 - precision: 0.8900 - 34ms/epoch - 4ms/step
Epoch 72/200
8/8 - 0s - loss: 0.2903 - accuracy: 0.8814 - recall: 0.8519 - precision: 0.8846 - 31ms/epoch - 4ms/step
Epoch 73/200
8/8 - 0s - loss: 0.2887 - accuracy: 0.8898 - recall: 0.8611 - precision: 0.8942 - 30ms/epoch - 4ms/step
Epoch 74/200
8/8 - 0s - loss: 0.2889 - accuracy: 0.8771 - recall: 0.8333 - precision: 0.8911 - 30ms/epoch - 4ms/step
Epoch 75/200
8/8 - 0s - loss: 0.2886 - accuracy: 0.8814 - recall: 0.8519 - precision: 0.8846 - 32ms/epoch - 4ms/step
Epoch 76/200
8/8 - 0s - loss: 0.2863 - accuracy: 0.8814 - recall: 0.8426 - precision: 0.8922 - 31ms/epoch - 4ms/step
Epoch 77/200
8/8 - 0s - loss: 0.2850 - accuracy: 0.8814 - recall: 0.8333 - precision: 0.9000 - 31ms/epoch - 4ms/step
Epoch 78/200
8/8 - 0s - loss: 0.2924 - accuracy: 0.8729 - recall: 0.8611 - precision: 0.8611 - 29ms/epoch - 4ms/step
Epoch 79/200
8/8 - 0s - loss: 0.2885 - accuracy: 0.8856 - recall: 0.8333 - pr

Epoch 141/200
8/8 - 0s - loss: 0.2252 - accuracy: 0.9110 - recall: 0.8981 - precision: 0.9065 - 30ms/epoch - 4ms/step
Epoch 142/200
8/8 - 0s - loss: 0.2267 - accuracy: 0.9153 - recall: 0.8611 - precision: 0.9490 - 31ms/epoch - 4ms/step
Epoch 143/200
8/8 - 0s - loss: 0.2266 - accuracy: 0.9195 - recall: 0.8981 - precision: 0.9238 - 34ms/epoch - 4ms/step
Epoch 144/200
8/8 - 0s - loss: 0.2238 - accuracy: 0.9153 - recall: 0.8981 - precision: 0.9151 - 32ms/epoch - 4ms/step
Epoch 145/200
8/8 - 0s - loss: 0.2228 - accuracy: 0.9153 - recall: 0.8704 - precision: 0.9400 - 32ms/epoch - 4ms/step
Epoch 146/200
8/8 - 0s - loss: 0.2245 - accuracy: 0.9025 - recall: 0.8426 - precision: 0.9381 - 32ms/epoch - 4ms/step
Epoch 147/200
8/8 - 0s - loss: 0.2222 - accuracy: 0.9195 - recall: 0.8889 - precision: 0.9320 - 31ms/epoch - 4ms/step
Epoch 148/200
8/8 - 0s - loss: 0.2167 - accuracy: 0.9153 - recall: 0.8981 - precision: 0.9151 - 32ms/epoch - 4ms/step
Epoch 149/200
8/8 - 0s - loss: 0.2212 - accuracy: 0.9110